# 04 · Validate — humanness↔stability trade-off, Vernier back-mutations, grafting vs resurfacing

**Standard slot:** *validate (in silico).* **For Project 16 the core analyses are:** (1) the
**humanness ↔ stability (ΔΔG) trade-off** across variants, (2) the effect of **Vernier-zone
back-mutations** (rescue stability at a humanness cost), and (3) the **CDR-grafting vs resurfacing**
comparison `[extension]` (D3 pt 2). This is what makes the project a *study*, not a demo.

Needs `results/campaign.csv`. All metrics on the mock backend are **SYNTHETIC** — the figures here
demonstrate the analysis; real numbers come from OASis/Hu-mAb + FoldX/Rosetta on an IgFold Fv model.

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · The humanness ↔ stability trade-off `[core]`

The headline result. Plot **humanness** (x) against the **ΔΔG proxy** (y, lower = more stable) for every
variant, coloured by method. The frontier you care about is the lower-right: **high humanness AND low
ΔΔG**. The parental control anchors low-humanness/zero-ΔΔG; bare grafts sit high-humanness/high-ΔΔG;
back-mutated grafts and resurfacing trade along the curve. **Mock values are SYNTHETIC** — the *shape*
and the *reasoning* are the deliverable.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

camp = pd.read_csv("results/campaign.csv")

fig, ax = plt.subplots(figsize=(7, 4.5))
markers = {"cdr_graft": "o", "resurface": "s", "parental": "*", "over_humanized_decoy": "X"}
for method, sub in camp.groupby("method"):
    ax.scatter(sub["oasis_like"], sub["ddg_kcal_mol"],
               marker=markers.get(method, "o"), s=70, label=method)
    for _, r in sub.iterrows():
        ax.annotate(str(r["variant_id"]).replace("EXAMPLE_DATA_", ""),
                    (r["oasis_like"], r["ddg_kcal_mol"]), fontsize=6, alpha=0.7)
ax.set_xlabel("humanness (OASis-like, heuristic — higher = more human)")
ax.set_ylabel("ΔΔG proxy (a.u.; >0 = destabilizing — lower = more stable)")
ax.set_title("Humanness vs stability trade-off — SYNTHETIC (mock)\nwant: lower-right (human AND stable)")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("results/tradeoff.png", dpi=150); plt.show()
print("Reminder: SYNTHETIC. On a real run use OASis/Hu-mAb humanness + FoldX/Rosetta ΔΔG.")

## 2 · Vernier-zone back-mutations — the rescue `[core]`

Back-mutations restore parental residues at framework positions that support the CDRs. They should
**lower ΔΔG** (regain stability/affinity) at a **small humanness cost**. Compare each bare graft to its
back-mutated counterpart; report the ΔΔG regained per humanness lost. This is the core decision a
humanization campaign makes: *which* Vernier residues to restore.

In [ ]:
from humanization_tools import (NONHUMAN_AB, HUMAN_FRAMEWORK, graft_cdrs, vernier_backmutations,
                                  score_variants, parental_sequence)

parent = parental_sequence(NONHUMAN_AB)

# Build the back-mutation ladder: graft, then add Vernier back-mutations one at a time, and watch
# humanness fall slightly while ΔΔG falls (rescue). SYNTHETIC values — the trend is the teaching point.
g = graft_cdrs(NONHUMAN_AB, HUMAN_FRAMEWORK, tool="mock")
bms = vernier_backmutations(g, NONHUMAN_AB, HUMAN_FRAMEWORK)
tokens = [b["token"] for b in bms]

ladder = []
for k in range(0, len(tokens) + 1):
    v = graft_cdrs(NONHUMAN_AB, HUMAN_FRAMEWORK, back_mutations=tuple(tokens[:k]), tool="mock",
                   variant_id=f"EXAMPLE_DATA_BMladder_{k}")
    score_variants([v], parent, tool="mock")
    ladder.append(dict(n_back_mutations=k, humanness=v.oasis_like, ddg_proxy=v.ddg_kcal_mol))
ladder_df = pd.DataFrame(ladder)
print("Back-mutation ladder (SYNTHETIC — direction is the teaching point):")
print(ladder_df.to_string(index=False))

fig, ax1 = plt.subplots(figsize=(6.5, 3.6))
ax1.plot(ladder_df["n_back_mutations"], ladder_df["humanness"], "o-", color="tab:blue", label="humanness")
ax1.set_xlabel("# Vernier back-mutations applied"); ax1.set_ylabel("humanness (heuristic)", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(ladder_df["n_back_mutations"], ladder_df["ddg_proxy"], "s--", color="tab:red", label="ΔΔG proxy")
ax2.set_ylabel("ΔΔG proxy (lower = more stable)", color="tab:red")
plt.title("Vernier back-mutations: stability rescue vs humanness cost (SYNTHETIC)")
plt.tight_layout(); plt.savefig("results/backmutation_ladder.png", dpi=150); plt.show()

## 3 · CDR grafting vs resurfacing `[extension]`

The two humanization strategies make a different bet. **CDR grafting** changes the whole framework
(high humanness, high ΔΔG risk, needs back-mutations). **Resurfacing** changes only surface framework
residues (low humanness gain, low ΔΔG risk). Compare them head-to-head on the same parental antibody:
which reaches your target humanness band at the lower stability cost? There is no universal winner — it
depends on the antibody and your humanness/ΔΔG bars.

In [ ]:
from humanization_tools import resurface

graft = graft_cdrs(NONHUMAN_AB, HUMAN_FRAMEWORK, tool="mock", variant_id="EXAMPLE_DATA_graft")
graft_bm = graft_cdrs(NONHUMAN_AB, HUMAN_FRAMEWORK, back_mutations=tuple(tokens), tool="mock",
                      variant_id="EXAMPLE_DATA_graft_BM")
veneer = resurface(NONHUMAN_AB, tool="mock", variant_id="EXAMPLE_DATA_resurface")
score_variants([graft, graft_bm, veneer], parent, tool="mock")

compare = pd.DataFrame([
    dict(strategy="CDR graft (bare)", humanness=graft.oasis_like, ddg_proxy=graft.ddg_kcal_mol,
         fr_muts=graft.n_framework_mutations),
    dict(strategy="CDR graft + Vernier BM", humanness=graft_bm.oasis_like, ddg_proxy=graft_bm.ddg_kcal_mol,
         fr_muts=graft_bm.n_framework_mutations),
    dict(strategy="Resurfacing", humanness=veneer.oasis_like, ddg_proxy=veneer.ddg_kcal_mol,
         fr_muts=veneer.n_framework_mutations),
])
print("Grafting vs resurfacing (SYNTHETIC mock):")
print(compare.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.scatter(compare["humanness"], compare["ddg_proxy"], s=90)
for _, r in compare.iterrows():
    ax.annotate(r["strategy"], (r["humanness"], r["ddg_proxy"]), fontsize=8,
                xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("humanness (heuristic)"); ax.set_ylabel("ΔΔG proxy (lower = more stable)")
ax.set_title("Grafting vs resurfacing — SYNTHETIC (want lower-right)")
ax.grid(alpha=0.3); plt.tight_layout(); plt.savefig("results/grafting_vs_resurfacing.png", dpi=150); plt.show()
print("Reminder: SYNTHETIC. No universal winner — report the trade-off for YOUR antibody with real tools.")

## 4 · Immunogenicity-risk summary (qualitative) `[core]`

Beyond the single humanness number, summarize the residual immunogenicity risk: how many framework
residues remain non-human, whether any back-mutations re-introduced non-human residues at exposed
positions, and whether known T-cell-epitope-prone motifs persist (a real run would add a T-cell epitope
predictor). Frame it as **risk**, not a guarantee — humanness scores correlate with, but do not prove,
low ADA.

In [ ]:
risk = []
for v_id, method, n_fr, n_bm, hum in zip(
        camp["variant_id"], camp["method"], camp["n_framework_mutations"],
        camp["n_back_mutations"], camp["oasis_like"]):
    band = "LOW" if hum >= 0.7 else ("MODERATE" if hum >= 0.5 else "HIGH")
    risk.append(dict(variant_id=v_id, method=method, humanness=hum,
                     residual_nonhuman_fr=int(n_fr) if pd.notna(n_fr) else None,
                     vernier_back_mutations=int(n_bm) if pd.notna(n_bm) else None,
                     immunogenicity_risk_band=band))
risk_df = pd.DataFrame(risk).sort_values("humanness", ascending=False)
risk_df.to_csv("results/immunogenicity_risk_summary.csv", index=False)
print("wrote results/immunogenicity_risk_summary.csv (SYNTHETIC bands — confirm with real tools + a")
print("T-cell-epitope predictor; humanness != guaranteed low ADA)")
risk_df

## D3 (part 2) checklist
- [ ] **Humanness ↔ ΔΔG trade-off** figure (the headline) with the frontier reasoning.
- [ ] **Vernier back-mutation** ladder: ΔΔG regained vs humanness lost; which residues to restore.
- [ ] **Grafting vs resurfacing** comparison `[extension]`; state which wins for *your* antibody and why.
- [ ] **Immunogenicity-risk summary** (residual non-human content + risk band), framed as risk.
- [ ] Every mock number labelled SYNTHETIC; conclusions phrased as plumbing/teaching, not results.

**Next:** `05_validation_plan.ipynb` — the ELISA/SPR/DSF validation plan + controls (parental +
over-humanized decoy).